In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import datetime

In [ ]:
rawData=pd.read_excel('R2_final_hotels.xlsx')

In [ ]:
data1=rawData[['ArrivalWeekNumber','ReservedRoom','AverageDailyRate','StaysInWeekendNights','StaysInWeekNights']]

In [ ]:
data1['Nights']=data1['StaysInWeekendNights']+data1['StaysInWeekNights']
data1['Revenue']=data1['Nights']*data1['AverageDailyRate']

In [ ]:
data2=data1.groupby(['ArrivalWeekNumber','ReservedRoom'])['Nights','Revenue'].sum(['Nights','Revenue'])

In [ ]:
data2['ADR']=data2['Revenue']/data2['Nights']

In [ ]:
data3=data2[['ADR']]

In [ ]:
for i in range(1,54):
  a=[]
  b=[]
  for j in ['A','B','C','D','E','F','G']:
    try:
      a.append(data3.loc[(i,j),'ADR'])
    except:
      b.append((i,j))
  for k in b:
    data3.loc[k,'ADR']=np.mean(a)

In [ ]:
data3=data3.groupby(['ArrivalWeekNumber','ReservedRoom']).sum(['ADR'])

In [ ]:
data3.reset_index().to_excel("ADRbyRoom&Week.xlsx")

In [ ]:
ADRbyRoom=pd.read_excel("ADRbyRoom&Week.xlsx").groupby(['ArrivalWeekNumber','ReservedRoom']).sum(['ADR'])

In [ ]:
def upgradeLoss(arrLike,roomADR):
  if (arrLike['ReservedRoom']==arrLike['AssignedRoom']) or arrLike['ReservationStatus']!='Check-Out':
    return 0
  else:
    try:
      return max((arrLike['StaysInWeekendNights']+arrLike['StaysInWeekNights'])*(roomADR.loc[(arrLike['ArrivalWeekNumber'],arrLike['AssignedRoom']),'ADR']-arrLike['AverageDailyRate']),0)
    except:
      return 'NA'

In [ ]:
def numFestivals(arrLike,df_festival):
  start_date=arrLike['ArrivalDate']
  end_date=arrLike['ReservationStatusDate']
  day_count = (end_date - start_date).days + 1
  festCount=0
  for single_date in [d for d in (start_date + datetime.timedelta(n) for n in range(day_count)) if d <= end_date]:
      festCount+=sum(df_festival[df_festival['Date']==datetime.datetime(2024, single_date.month, single_date.day)]['Festivals'])
  return festCount

In [ ]:
dataClean=rawData.copy(deep=True)
festival=pd.read_excel("Festivals.xlsx")

In [ ]:
dataClean['NumFestivals']=dataClean.apply(numFestivals,axis=1,df_festival=festival)
dataClean['RevenueLossbyUpgrade']=rawData.apply(upgradeLoss,axis=1,roomADR=ADRbyRoom)

In [ ]:
dataClean['RevenuebyStays']=(dataClean['StaysInWeekendNights']+dataClean['StaysInWeekNights'])*dataClean['AverageDailyRate']

In [ ]:
def cancelLost(arrLike):
  if arrLike['ReservationStatus']=='Canceled' and ((arrLike['ArrivalDate']-arrLike['ReservationStatusDate']).days<=3):
    return max((arrLike['StaysInWeekendNights']+arrLike['StaysInWeekNights']-1)*arrLike['AverageDailyRate'],0)
  else:
    return 0

In [ ]:
dataClean['RevenueLossbyCancel']=rawData.apply(cancelLost,axis=1)

In [ ]:
dataClean=dataClean.drop(columns='BookingCompany')

In [ ]:
dataClean=dataClean.dropna()

In [ ]:
dataClean['PotentialRevenue']=dataClean['RevenuebyStays']-dataClean['RevenueLossbyUpgrade']-dataClean['RevenueLossbyCancel']

In [ ]:
dataClean.to_excel("R2_final_hotels_Clean.xlsx",index=False)